In [1]:
# install dependencies first
!pip -q install dlt[duckdb]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.9/348.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 2.1 MB/s eta 0:00:00


In [2]:
import dlt
import dlt
from itertools import islice
from dlt.sources.rest_api import rest_api_source

In [3]:
def openlibrary_source(query: str = "harry potter"):

    return rest_api_source({
        "client": {
            "base_url": "https://openlibrary.org",
        },
        "resource_defaults": {
            "primary_key": "key",
            "write_disposition": "replace",
        },
        "resources": [
            {
                "name": "books",
                "endpoint": {
                    "path": "search.json",
                    "params": {
                        "q": query,
                        "limit": 100,
                    },
                    "data_selector": "docs",
                    "paginator": {
                        "type": "offset",
                        "limit": 100,
                        "offset_param": "offset",
                        "limit_param": "limit",
                        "total_path": "numFound",
                    },
                },
            },
        ],
    })


In [4]:
pipeline = dlt.pipeline(
    pipeline_name="ol_demo",
    destination="duckdb",
    dataset_name="ol_data",
    progress="log" # logs the pipeline run (Optiona)
)

In [6]:
extract_info = pipeline.extract(openlibrary_source())


------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 223.03 MB (9.70%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.69s | Rate: 0.00/s
books: 100  | Time: 0.00s | Rate: 6765006.45/s
Memory usage: 223.35 MB (9.80%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 2.10s | Rate: 0.00/s
books: 400  | Time: 1.42s | Rate: 282.45/s
Memory usage: 223.90 MB (9.80%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 3.28s | Rate: 0.00/s
books: 500  | Time: 2.60s | Rate: 192.48/s
Memory usage: 224.30 MB (9.80%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 4.65s | Rate: 0.00/s

In [7]:
load_id = extract_info.loads_ids[-1]
m = extract_info.metrics[load_id][0]

print("Resources:", list(m["resource_metrics"].keys()))
print("Tables:", list(m["table_metrics"].keys()))
print("Load ID:", load_id)
print()

for resource, rm in m["resource_metrics"].items():
    print(f"Resource: {resource}")
    print(f"rows extracted: {rm.items_count}")
    print()

Resources: ['books']
Tables: ['books']
Load ID: 1772455213.422141

Resource: books
rows extracted: 3759



In [8]:
normalize_info = pipeline.normalize()

------------------- Normalize rest_api in 1772455213.422141 --------------------
Files: 0/2 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 239.40 MB (12.70%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1772455213.422141 --------------------
Files: 0/2 (0.0%) | Time: 0.00s | Rate: 0.00/s
Items: 0  | Time: 0.00s | Rate: 0.00/s
Memory usage: 239.40 MB (12.70%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1772455213.422141 --------------------
Files: 10/2 (500.0%) | Time: 1.19s | Rate: 8.38/s
Items: 0  | Time: 1.19s | Rate: 0.00/s
Memory usage: 248.75 MB (12.80%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1772455213.422141 --------------------
Files: 10/2 (500.0%) | Time: 1.20s | Rate: 8.34/s
Items: 23083  | Time: 1.20s | Rate: 19267.57/s
Memory usage: 248.75 MB (12.80%) | CPU usage: 0.00%



In [9]:
load_id = normalize_info.loads_ids[-1]
m = normalize_info.metrics[load_id][0]

print("Load ID:", load_id)
print()

print("Tables created/updated:")
for table_name, tm in m["table_metrics"].items():
    # skip dlt internal tables to keep it beginner-friendly
    if table_name.startswith("_dlt"):
        continue
    print(f"  - {table_name}: {tm.items_count} rows")


Load ID: 1772455213.422141

Tables created/updated:
  - books: 3759 rows
  - books__author_key: 4637 rows
  - books__author_name: 4637 rows
  - books__ia: 3434 rows
  - books__ia_collection: 2735 rows
  - books__language: 3748 rows
  - books__id_standard_ebooks: 12 rows
  - books__id_librivox: 64 rows
  - books__id_project_gutenberg: 56 rows


In [10]:
pipeline.default_schema

<dlt.Schema(name='rest_api', version=2, tables=['_dlt_version', '_dlt_loads', 'books', '_dlt_pipeline_state', 'books__author_key', 'books__author_name', 'books__ia', 'books__ia_collection', 'books__language', 'books__id_standard_ebooks', 'books__id_librivox', 'books__id_project_gutenberg'], version_hash='ZJIabaQJ9DAYgsR04wEVeXOgU80roBUfdvrR2YoBEyU=')>

In [11]:
load_info = pipeline.load()

---------------------- Load rest_api in 1772455213.422141 ----------------------
Jobs: 0/10 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 260.77 MB (10.40%) | CPU usage: 0.00%

---------------------- Load rest_api in 1772455213.422141 ----------------------
Jobs: 0/10 (0.0%) | Time: 1.25s | Rate: 0.00/s
Memory usage: 322.34 MB (10.80%) | CPU usage: 0.00%

---------------------- Load rest_api in 1772455213.422141 ----------------------
Jobs: 5/10 (50.0%) | Time: 2.46s | Rate: 2.04/s
Memory usage: 356.07 MB (11.20%) | CPU usage: 0.00%

---------------------- Load rest_api in 1772455213.422141 ----------------------
Jobs: 10/10 (100.0%) | Time: 3.48s | Rate: 2.87/s
Memory usage: 292.91 MB (10.70%) | CPU usage: 0.00%



In [12]:
load_info = pipeline.run(openlibrary_source())

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 290.07 MB (10.60%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.54s | Rate: 0.00/s
books: 100  | Time: 0.00s | Rate: 11037642.11/s
Memory usage: 290.07 MB (10.70%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 1.77s | Rate: 0.00/s
books: 400  | Time: 1.23s | Rate: 325.44/s
Memory usage: 290.07 MB (10.80%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 3.13s | Rate: 0.00/s
books: 700  | Time: 2.59s | Rate: 270.76/s
Memory usage: 290.19 MB (10.80%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 4.40s | Rate: 0

In [13]:
ds = pipeline.dataset()

In [14]:
ds.tables

['books',
 'books__author_key',
 'books__author_name',
 'books__ia',
 'books__ia_collection',
 'books__language',
 'books__id_standard_ebooks',
 'books__id_librivox',
 'books__id_project_gutenberg',
 '_dlt_version',
 '_dlt_loads',
 '_dlt_pipeline_state']

In [15]:
df = ds.books.df()

In [16]:
df

,cover_edition_key,cover_i,ebook_access,edition_count,first_publish_year,has_fulltext,key,lending_edition_s,lending_identifier_s,public_scan_b,title,_dlt_load_id,_dlt_id,subtitle
0,OL61027601M,15155833,borrowable,397,1997,True,/works/OL82563W,OL38565767M,harrypotterylapi0000rowl_q5r6,False,Harry Potter and the Philosopher's Stone,1772455757.9569042,yhM5ZqZtfI1Qtw,None
1,OL26378158M,15158660,printdisabled,144,2007,True,/works/OL82586W,None,None,False,Harry Potter and the Deathly Hallows,1772455757.9569042,nROrmsPUi2+xWQ,None
2,OL26234270M,10580435,borrowable,279,1999,True,/works/OL82536W,OL48101764M,bdrc-W8LS66814,False,Harry Potter and the Prisoner of Azkaban,1772455757.9569042,1QuUpTHF+8EW/A,None
3,OL25683482M,12059372,printdisabled,242,2000,True,/works/OL82560W,None,None,False,Harry Potter and the Goblet of Fire,1772455757.9569042,5nr+g/i52eSHcA,None
4,OL25662116M,15158666,printdisabled,247,2003,True,/works/OL82548W,None,None,False,Harry Potter and the Order of the Phoenix,1772455757.9569042,IO+QCLrCSsvG8g,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3754,None,<NA>,no_ebook,2,1660,False,/works/OL32973222W,None,None,False,The Holy Bible,1772455757.9569042,w0qcHjFsAoBtyw,None
3755,None,<NA>,no_ebook,1,<NA>,False,/works/OL16130757W,None,None,False,Earl Warren papers,1772455757.9569042,F7SRdQeS2aAQAg,None
3756,OL1001200M,4529961,borrowable,1,1997,True,/works/OL3338390W,OL1001200M,designofwilliamm0000boos,False,T.S. Eliot's use of popular sources,1772455757.9569042,XUw6NspOR3N9jg,None
3757,None,<NA>,no_ebook,1,<NA>,False,/works/OL16511044W,None,None,False,Russell Wheeler Davenport papers,1772455757.9569042,+se1STYMvuy0mQ,None
